In [0]:
from typing import Dict

from pyspark.sql import DataFrame
from pyspark.sql.functions import *

In [0]:
class BronzeETL:
    """Bronze layer ETL for Yelp data ingestion in Databricks"""

    def __init__(self):

        # Databricks already provides the SparkSession
        self.spark = spark

        # Raw Yelp data is stored in the Unity Catalog Volume
        self.yelp_data_path = "/Volumes/yelp_dataset/default/raw_yelp"

        # Bronze layer is stored as Unity Catalog tables
        self.bronze_catalog = "yelp_dataset"
        self.bronze_schema = "bronze"

    # ------------------------------------------------------------------
    # Extract
    # ------------------------------------------------------------------

    def extract_yelp_tar(self) -> bool:
        """
        Original project extracted the Yelp tar file locally.

        In Databricks this is no longer required because the Yelp JSON
        files have already been transferred from Google Drive into the
        Unity Catalog Volume.

        This method is retained so the structure of the original
        project remains unchanged.
        """

        try:

            files = dbutils.fs.ls(self.yelp_data_path)

            json_files = [
                file.path
                for file in files
                if file.path.lower().endswith(".json")
            ]

            if json_files:

                print(
                    f"Yelp JSON files found in "
                    f"{self.yelp_data_path}"
                )

                for file in json_files:
                    print(f"  {file}")

                return True

            print(
                f"No Yelp JSON files found in "
                f"{self.yelp_data_path}"
            )

            return False

        except Exception as e:

            print(
                f"Failed to access Yelp data path: {e}"
            )

            return False

    # ------------------------------------------------------------------
    # Get Yelp files
    # ------------------------------------------------------------------

    def get_yelp_files(self) -> Dict[str, str]:
        """
        Get paths to Yelp JSON files from the Unity Catalog Volume.

        This replaces the local pathlib-based file discovery from the
        original project.
        """

        files = {
            "business": None,
            "review": None,
            "user": None,
            "checkin": None,
            "tip": None
        }

        try:

            volume_files = dbutils.fs.ls(
                self.yelp_data_path
            )

            for file_info in volume_files:

                if file_info.isDir():
                    continue

                filename = file_info.name.lower()

                if (
                    "business" in filename
                    and files["business"] is None
                ):
                    files["business"] = file_info.path

                elif (
                    "review" in filename
                    and files["review"] is None
                ):
                    files["review"] = file_info.path

                elif (
                    "user" in filename
                    and files["user"] is None
                ):
                    files["user"] = file_info.path

                elif (
                    "checkin" in filename
                    and files["checkin"] is None
                ):
                    files["checkin"] = file_info.path

                elif (
                    "tip" in filename
                    and files["tip"] is None
                ):
                    files["tip"] = file_info.path

        except Exception as e:

            print(
                f"Failed to find Yelp files: {e}"
            )

        return files

    # ------------------------------------------------------------------
    # Business
    # ------------------------------------------------------------------

    def ingest_business_data(
        self,
        file_path: str
    ) -> DataFrame:

        """Ingest Yelp business data"""

        print(
            f"Ingesting business data from {file_path}"
        )

        # Yelp JSON is JSON Lines format
        df = (
            self.spark.read
            .option("multiLine", "false")
            .json(file_path)
        )

        # Add metadata columns
        df = (
            df
            .withColumn(
                "ingestion_timestamp",
                current_timestamp()
            )
            .withColumn(
                "source_file",
                lit(file_path.split("/")[-1])
            )
            .withColumn(
                "data_layer",
                lit("bronze")
            )
        )

        # Filter for restaurants only
        df = df.filter(
            col("categories").isNotNull()
            &
            (
                col("categories").contains("Restaurant")
                |
                col("categories").contains("Food")
                |
                col("categories").contains("Dining")
            )
        )

        print(
            f"Business data ingested: "
            f"{df.count()} records"
        )

        return df

    # ------------------------------------------------------------------
    # Review
    # ------------------------------------------------------------------

    def ingest_review_data(
        self,
        file_path: str
    ) -> DataFrame:

        """Ingest Yelp review data"""

        print(
            f"Ingesting review data from {file_path}"
        )

        df = (
            self.spark.read
            .option("multiLine", "false")
            .json(file_path)
        )

        # Add metadata columns
        df = (
            df
            .withColumn(
                "ingestion_timestamp",
                current_timestamp()
            )
            .withColumn(
                "source_file",
                lit(file_path.split("/")[-1])
            )
            .withColumn(
                "data_layer",
                lit("bronze")
            )
        )

        print(
            f"Review data ingested: "
            f"{df.count()} records"
        )

        return df

    # ------------------------------------------------------------------
    # User
    # ------------------------------------------------------------------

    def ingest_user_data(
        self,
        file_path: str
    ) -> DataFrame:

        """Ingest Yelp user data"""

        print(
            f"Ingesting user data from {file_path}"
        )

        df = (
            self.spark.read
            .option("multiLine", "false")
            .json(file_path)
        )

        # Add metadata columns
        df = (
            df
            .withColumn(
                "ingestion_timestamp",
                current_timestamp()
            )
            .withColumn(
                "source_file",
                lit(file_path.split("/")[-1])
            )
            .withColumn(
                "data_layer",
                lit("bronze")
            )
        )

        print(
            f"User data ingested: "
            f"{df.count()} records"
        )

        return df

    # ------------------------------------------------------------------
    # Checkin
    # ------------------------------------------------------------------

    def ingest_checkin_data(
        self,
        file_path: str
    ) -> DataFrame:

        """Ingest Yelp checkin data"""

        print(
            f"Ingesting checkin data from {file_path}"
        )

        df = (
            self.spark.read
            .option("multiLine", "false")
            .json(file_path)
        )

        # Add metadata columns
        df = (
            df
            .withColumn(
                "ingestion_timestamp",
                current_timestamp()
            )
            .withColumn(
                "source_file",
                lit(file_path.split("/")[-1])
            )
            .withColumn(
                "data_layer",
                lit("bronze")
            )
        )

        print(
            f"Checkin data ingested: "
            f"{df.count()} records"
        )

        return df

    # ------------------------------------------------------------------
    # Tip
    # ------------------------------------------------------------------

    def ingest_tip_data(
        self,
        file_path: str
    ) -> DataFrame:

        """Ingest Yelp tip data"""

        print(
            f"Ingesting tip data from {file_path}"
        )

        df = (
            self.spark.read
            .option("multiLine", "false")
            .json(file_path)
        )

        # Add metadata columns
        df = (
            df
            .withColumn(
                "ingestion_timestamp",
                current_timestamp()
            )
            .withColumn(
                "source_file",
                lit(file_path.split("/")[-1])
            )
            .withColumn(
                "data_layer",
                lit("bronze")
            )
        )

        print(
            f"Tip data ingested: "
            f"{df.count()} records"
        )

        return df

    # ------------------------------------------------------------------
    # Save to Bronze
    # ------------------------------------------------------------------

    def save_to_bronze(
        self,
        df: DataFrame,
        table_name: str
    ):

        """
        Save DataFrame to the Databricks Bronze layer.

        Original project:
            DataFrame → Parquet directory

        Databricks:
            DataFrame → Delta table in Unity Catalog
        """

        full_table_name = (
            f"{self.bronze_catalog}."
            f"{self.bronze_schema}."
            f"{table_name}"
        )

        print(
            f"Saving {table_name} to "
            f"{full_table_name}"
        )

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option(
                "overwriteSchema",
                "true"
            )
            .saveAsTable(full_table_name)
        )

        print(
            f"Saved {table_name} to Bronze layer"
        )

    # ------------------------------------------------------------------
    # Run
    # ------------------------------------------------------------------

    def run(self) -> bool:

        """Run complete Bronze layer ETL"""

        print(
            "Starting Bronze Layer ETL"
        )

        # --------------------------------------------------------------
        # Check Yelp data in Databricks Volume
        # --------------------------------------------------------------

        if not self.extract_yelp_tar():

            return False

        # --------------------------------------------------------------
        # Get file paths
        # --------------------------------------------------------------

        yelp_files = self.get_yelp_files()

        print(
            "\nYelp files discovered:"
        )

        for dataset, file_path in yelp_files.items():

            print(
                f"{dataset}: {file_path}"
            )

        # --------------------------------------------------------------
        # Ingest each dataset
        # --------------------------------------------------------------

        datasets = {}

        # --------------------------------------------------------------
        # Business
        # --------------------------------------------------------------

        if yelp_files["business"]:

            datasets["business"] = (
                self.ingest_business_data(
                    yelp_files["business"]
                )
            )

            self.save_to_bronze(
                datasets["business"],
                "yelp_business"
            )

        # --------------------------------------------------------------
        # Review
        # --------------------------------------------------------------

        if yelp_files["review"]:

            datasets["review"] = (
                self.ingest_review_data(
                    yelp_files["review"]
                )
            )

            self.save_to_bronze(
                datasets["review"],
                "yelp_review"
            )

        # --------------------------------------------------------------
        # User
        # --------------------------------------------------------------

        if yelp_files["user"]:

            datasets["user"] = (
                self.ingest_user_data(
                    yelp_files["user"]
                )
            )

            self.save_to_bronze(
                datasets["user"],
                "yelp_user"
            )

        # --------------------------------------------------------------
        # Checkin
        # --------------------------------------------------------------

        if yelp_files["checkin"]:

            datasets["checkin"] = (
                self.ingest_checkin_data(
                    yelp_files["checkin"]
                )
            )

            self.save_to_bronze(
                datasets["checkin"],
                "yelp_checkin"
            )

        # --------------------------------------------------------------
        # Tip
        # --------------------------------------------------------------

        if yelp_files["tip"]:

            datasets["tip"] = (
                self.ingest_tip_data(
                    yelp_files["tip"]
                )
            )

            self.save_to_bronze(
                datasets["tip"],
                "yelp_tip"
            )

        # --------------------------------------------------------------
        # Completed
        # --------------------------------------------------------------

        print(
            "Bronze Layer ETL completed successfully"
        )

        return True


# ==========================================================================
# Run Bronze ETL
# ==========================================================================

bronze_etl = BronzeETL()

bronze_etl.run()

In [0]:
# bronze_etl = BronzeETL()
# result = bronze_etl.get_yelp_files()

In [0]:
# List all files in the raw_yelp volume
files = dbutils.fs.ls("/Volumes/yelp_dataset/default/raw_yelp")

print("Files in raw_yelp volume:")
print("=" * 80)
for file_info in files:
    file_type = "DIR" if file_info.isDir() else "FILE"
    size_mb = file_info.size / (1024 * 1024)  # Convert to MB
    print(f"{file_type:6} | {file_info.name:50} | {size_mb:>10.2f} MB")

In [0]:
YELP_FILES = {
    "business": "yelp_academic_dataset_business.json",
    "review": "yelp_academic_dataset_review.json",
    "user": "yelp_academic_dataset_user.json",
    "checkin": "yelp_academic_dataset_checkin.json",
    "tip": "yelp_academic_dataset_tip.json"
}

In [0]:
YELP_PATHS = {
    name: f"{RAW_PATH}/{filename}"
    for name, filename in YELP_FILES.items()
}

YELP_PATHS